# System prompts

A system prompt sets the role, tone and rules for the whole conversation.
It goes in the separate `system` parameter, not in `messages`.

- `client.messages.create(..., system="You are ...")`
- the API rejects `system=None`, so `chat()` only adds it when it is set

## Setup

The client reads `ANTHROPIC_API_KEY` (and `ANTHROPIC_BASE_URL`, if set) from `.env` in the repo root.

In [1]:
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

## Helpers: `chat()` gets an optional `system`

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return "".join(block.text for block in message.content if block.type == "text")

## Same question, with and without a system prompt

In [3]:
messages = []
add_user_message(messages, "How do I solve 5x + 3 = 2x + 9 for x?")

# Without system prompt
answer = chat(messages)
print(answer)

# Solving 5x + 3 = 2x + 9

**Step 1:** Subtract 2x from both sides
- 5x - 2x + 3 = 9
- 3x + 3 = 9

**Step 2:** Subtract 3 from both sides
- 3x = 6

**Step 3:** Divide both sides by 3
- x = 2

**Check:** 5(2) + 3 = 13 and 2(2) + 9 = 13 ✓

**Answer: x = 2**


In [4]:
# With system prompt
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

answer = chat(messages, system=system)
print(answer)

Great question! Let me guide you through this step-by-step.

**Step 1:** What's your goal here?
Think about what you want to end up with — you want to get x by itself on one side of the equation.

**Step 2:** Look at both sides of your equation:
- Left side: 5x + 3
- Right side: 2x + 9

Where do you see x terms? Can you get all the x terms on one side by moving one of them?

*Hint: Try subtracting 2x from both sides. What do you get?*

Once you do that, let me know what your new equation looks like, and we'll keep going!


Without the system prompt Claude solves the equation right away.
With it, Claude acts as a tutor: it asks guiding questions instead of giving the answer.